<a href="https://colab.research.google.com/github/SchwiLynn/ACM_RESEARCH_2026-Training-a-chess-model/blob/main/notebooks/colab_setup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ACM Research 2026 - Colab Setup

Track 1: personalize Maia-2 on a single player's games.

**Run order:** cells 1-4 every new session (Colab wipes everything on disconnect).
Cells 5+ only when you need them.

Runtime: **CPU** for data prep and evaluation. Switch to **T4 GPU** only for fine-tuning.

## 1. Environment check

In [ ]:
!nvidia-smi
import sys
print(sys.version)
# maia2 supports Python 3.10-3.12. If Colab is on 3.13+, pip install will fail
# and you need a pinned env instead. Check this FIRST, not after a cryptic error.

## 2. Mount Drive (persistent storage)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJ = '/content/drive/MyDrive/acm_chess'
!mkdir -p {PROJ}/data {PROJ}/checkpoints {PROJ}/runs
print(PROJ)

## 3. Install


In [ ]:
%cd /content
![ -d repo ] || git clone https://github.com/SchwiLynn/ACM_RESEARCH_2026-Training-a-chess-model.git repo
%cd /content/repo
!git pull -q
!pip install -q -r requirements.txt

D = f'{PROJ}/data'
!mkdir -p {D}/raw {D}/moves {D}/splits
!ln -sfn {D}/raw data/raw
!ln -sfn {D}/moves data/moves
!ln -sfn {D}/splits data/splits

## 4. Get the data

Lichess handed over data as PGN, over the API.


In [ ]:
# Only needed once: data is already in Drive after the first run.
# Test on one player first, then run everyone. Already-downloaded players are skipped.
!echo thakt > /tmp/one.txt
!python shared/p5_prepare_all_players.py /tmp/one.txt
# !python shared/p5_prepare_all_players.py config/eval_players.txt

## 6. Load Maia-2 and confirm it runs

In [ ]:
from maia2 import dataset, inference, model

maia2_model = model.from_pretrained(type="rapid", device="auto")
data = dataset.load_example_test_dataset()

data, accuracy = inference.inference_batch(
    data,
    maia2_model,
    verbose=1,
    batch_size=1024,
    num_workers=0,
)
print('baseline move-matching accuracy on example set:', accuracy)

## 8. Checkpoint pattern

Colab disconnects. Save every epoch to Drive.

In [ ]:
import torch, json, time

def save_ckpt(model, optimizer, epoch, run_name, metrics=None):
    path = f'{PROJ}/checkpoints/{run_name}_epoch{epoch:02d}.pt'
    torch.save({
        'epoch': epoch,
        'model': model.state_dict(),
        'opt': optimizer.state_dict(),
        'metrics': metrics or {},
        'ts': time.time(),
    }, path)
    with open(f'{PROJ}/runs/{run_name}.jsonl', 'a') as f:
        f.write(json.dumps({'epoch': epoch, **(metrics or {})}) + '\n')
    return path

## 9. Freeze the environment for the writeup

Run this at the end of any run whose numbers go in the report.

In [ ]:
!pip freeze > {PROJ}/runs/requirements.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv > {PROJ}/runs/gpu.txt
print('frozen')